cd /d/Code/LLM && uv add openai gradio

In [ ]:
# 导入依赖
import os
from openai import OpenAI
import gradio as gr

# 配置环境
client = OpenAI(
    api_key = "",
    base_url = ""
)

In [3]:
# 系统提示词
systemprompt="""
你是一个专业的***。拥有专业经验。请根据***。请用清晰的结构组织回复，让建议易于阅读和实施。
示例：****
"""

# 按钮绑定函数（将参数传入用户提示词中，并调用模型）
def recommender(qa1, qa2, qa3):
    userprompt = f"""
    Q: 问题1
    A: {qa1}
    Q: 问题2
    A: {qa2}
    Q: 问题3
    A: {qa3}
    """
    # 调用模型
    response = client.chat.completions.create(
        model = "",
        messages = [
            {"role": "system", "content": systemprompt},
            {"role": "user", "content": userprompt}
        #   {"role": "assistant", "content": "让 AI 记住上一句说了啥，把 AI 上一句的回复也打包进 messages 列表里传给它,多轮对话中必不可少"}
        ],
        stream = True,
        temperature = 0.5
    )
    # 非流式 
    # return response.choices[0].message.content
    # 处理流式响应
    result = " "
    for chunk in response:
        if chunk.choices[0].delta.content is not None:
            result += chunk.choices[0].delta.content
            yield result

In [4]:
# 创建 Gridio 界面（自定义的空白画板，起名叫 demo）
with gr.Blocks() as demo:
    # 设置标题
    gr.Markdown("# 🍳 智能****系统")
    gr.Markdown("### 为您量身定制个性化****，请根据一下问题提供信息：")

    # 问题列表
    question_list = [
        "1、问题1",
        "2、问题2",
        "3、问题3"
    ]

    # 创建输入框
    q1_input = gr.Textbox(label=question_list[0])           # 在输入框上方显示“1、问题1”
    q2_input = gr.Textbox(label=question_list[1])
    q3_input = gr.Textbox(label=question_list[2])

    # 创建按钮
    submit = gr.Button("生成**推荐")

    # 创建输出框
    result = gr.Textbox(
        label = "📝 您的个性化**",                                        # 在输出框上方显示“**”
        placeholder = "点击生成按钮后，详细的**推荐将显示在这里...",        # 灰色提示字
        lines = 20
    )

    # 绑定按钮点击事件
    submit.click(
        recommender,
        inputs = [q1_input, q2_input, q3_input],
        outputs = result
    )

    # 启动应用
    demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
